# Descritive

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")
import pessoal
from pessoal import *

spark = SparkSession.builder \
    .appName("SIM Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [5]:
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline'
ibge_path = f"{PATH}/data/processed/ibge_codigos/02_standardization/"
ibge_codigos = spark.read.parquet(ibge_path)

In [6]:
pessoal.completudeSchema(ibge_codigos)

[Stage 1:>                                                          (0 + 1) / 1]

Qtd. registros: 5571 | Quantidade de colunas:  9
root
 |-- cod_uf: long (nullable = true)
 |-- nm_uf: string (nullable = true)
 |-- reg_geo_inter: long (nullable = true)
 |-- nm_reg_geo_inter: string (nullable = true)
 |-- reg_geo_imed: long (nullable = true)
 |-- nm_reg_geo_imed: string (nullable = true)
 |-- municipio: long (nullable = true)
 |-- cod_mun_comp: long (nullable = true)
 |-- nm_mun: string (nullable = true)



In [9]:
pessoal.completudeVar(ibge_codigos)

Resumo do DataFrame:


,coluna,total,duplicados,nulos,valores_unicos
0,cod_uf,5571,5544,0,27
1,nm_uf,5571,5544,0,27
2,reg_geo_inter,5571,5438,0,133
3,nm_reg_geo_inter,5571,5438,0,133
4,nm_reg_geo_imed,5571,5063,0,508
5,reg_geo_imed,5571,5061,0,510
6,municipio,5571,1263,0,4308
7,nm_mun,5571,273,0,5298
8,cod_mun_comp,5571,0,0,5571


In [10]:
for col in ibge_codigos.columns:
    ibge_codigos.groupBy(col).count().show()

+------+-----+
|cod_uf|count|
+------+-----+
|    26|  185|
|    29|  417|
|    22|  224|
|    50|   79|
|    32|   78|
|    43|  497|
|    31|  853|
|    25|  223|
|    27|  102|
|    51|  142|
|    52|  246|
|    17|  139|
|    41|  399|
|    28|   75|
|    33|   92|
|    12|   22|
|    11|   52|
|    35|  645|
|    13|   62|
|    14|   15|
+------+-----+
only showing top 20 rows
+-------------------+-----+
|              nm_uf|count|
+-------------------+-----+
|     Santa Catarina|  295|
| Mato Grosso do Sul|   79|
|              Goiás|  246|
|        Mato Grosso|  142|
|              Ceará|  184|
|              Piauí|  224|
|     Espírito Santo|   78|
|             Paraná|  399|
|            Alagoas|  102|
|              Bahia|  417|
|            Roraima|   15|
|   Distrito Federal|    1|
|         Pernambuco|  185|
|           Amazonas|   62|
|               Acre|   22|
|  Rio Grande do Sul|  497|
|Rio Grande do Norte|  167|
|            Sergipe|   75|
|          São Paulo|  645|

In [11]:
numeric_cols = [
    c for c,t in ibge_codigos.dtypes
    if t in ["int","bigint","double","float"]
]

In [12]:
ibge_codigos.select(numeric_cols).summary().show()

[Stage 277:>                                                        (0 + 1) / 1]

+-------+------------------+------------------+------------------+------------------+-----------------+
|summary|            cod_uf|     reg_geo_inter|      reg_geo_imed|         municipio|     cod_mun_comp|
+-------+------------------+------------------+------------------+------------------+-----------------+
|  count|              5571|              5571|              5571|              5571|             5571|
|   mean|32.381080595943274|3242.1866810267456|323826.88835038594|15814.473164602405| 3253922.53275893|
| stddev| 9.836143785913537| 984.0413521484692| 98363.45569952259|15996.960239526155|985133.1883166808|
|    min|                11|              1101|            110001|                13|          1100015|
|    25%|                25|              2503|            250009|              4506|          2512077|
|    50%|                31|              3107|            310039|             10400|          3146255|
|    75%|                41|              4104|            41001